# 01 — Entendimento dos dados


**Objetivo:** conhecer a base de dados

### Origem e construção da base

- Fonte: [Geração por usina — ONS](https://dados.ons.org.br/dataset/geracao-usina-2).
- [Dicionário de dados do ONS](https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/geracao_usina_2_ho/DicionarioDados_GeracaoPorUsina.pdf).
- O manifesto recebido relaciona 24 recursos mensais, de janeiro de 2024 a dezembro de 2025.
- Foi escolhido três usinas de Pernambuco
- **Recorte inicial desse notebook:** Conj. Belmonte, identificador `CJU_PEBELM`, janeiro de 2024.

O manifesto (\data\raw\manifesto_downloads_ons.csv) registra as URLs e a quantidade de linhas após o filtro inicial de PE. O código que gerou solar_pe_3_usinas_2024_2025.csv foi utilizado externo ao projeto e, até o momento da edição deste notebook, não foi adicionado ao escopo. Porém, nenhum dado foi criado, todos foram retirados da base da ONS.

In [4]:
# ■■■■■■■■■■■■■■ BIBLIOTECAS ■■■■■■■■■■■■■■
import sys
import pandas as pd
import json
import datetime as dt

In [ ]:
print(sys.executable)

## 1. Leitura dos dados


In [ ]:
# ------ Localiza a pasta do projeto
from pathlib import Path

pasta_atual = Path.cwd()
candidatas = [pasta_atual, *pasta_atual.parents]

pasta_projeto = next(
    (pasta for pasta in candidatas
     if (pasta / 'data' / 'raw' / 'solar_pe_3_usinas_2024_2025.csv').is_file()),
    None
)

if pasta_projeto is None:
    raise FileNotFoundError(
        'CSV não encontrado. Abra a pasta P1 no VS Code e confira se o '
        'arquivo solar_pe_3_usinas_2024_2025.csv está em data/raw/.'
    )

pasta_dados = pasta_projeto / 'data' / 'raw'
caminho = pasta_dados / 'solar_pe_3_usinas_2024_2025.csv'
print('Pasta do projeto:', pasta_projeto)

In [5]:
# ------ Leitura dos dados
dados = pd.read_csv(caminho, sep=';')

display(dados.head())

,din_instante,id_subsistema,nom_subsistema,id_estado,nom_estado,cod_modalidadeoperacao,nom_tipousina,nom_tipocombustivel,nom_usina,id_ons,ceg,val_geracao
0,2024-01-01 00:00:00,NE,NORDESTE,PE,PERNAMBUCO,Conjunto de Usinas,FOTOVOLTAICA,Fotovoltaica,Conj. Belmonte,CJU_PEBELM,-,0.0
1,2024-01-01 01:00:00,NE,NORDESTE,PE,PERNAMBUCO,Conjunto de Usinas,FOTOVOLTAICA,Fotovoltaica,Conj. Belmonte,CJU_PEBELM,-,0.0
2,2024-01-01 02:00:00,NE,NORDESTE,PE,PERNAMBUCO,Conjunto de Usinas,FOTOVOLTAICA,Fotovoltaica,Conj. Belmonte,CJU_PEBELM,-,0.0
3,2024-01-01 03:00:00,NE,NORDESTE,PE,PERNAMBUCO,Conjunto de Usinas,FOTOVOLTAICA,Fotovoltaica,Conj. Belmonte,CJU_PEBELM,-,0.0
4,2024-01-01 04:00:00,NE,NORDESTE,PE,PERNAMBUCO,Conjunto de Usinas,FOTOVOLTAICA,Fotovoltaica,Conj. Belmonte,CJU_PEBELM,-,0.0


In [11]:
# ------ Identificação do arquivo utilizado
import hashlib

hash_sha256 = hashlib.sha256()
with caminho.open('rb') as arquivo:
    for bloco in iter(lambda: arquivo.read(1024 * 1024), b''):
        hash_sha256.update(bloco)

print('Arquivo:', caminho.name)
print('SHA-256:', hash_sha256.hexdigest())


Arquivo: solar_pe_3_usinas_2024_2025.csv
SHA-256: 929c2731942bfd4fd814044620d4d1283dd8dcb9cabb94eeeadc9d984d71d119


O hash identifica os bytes do CSV usado nesta execução.

## 2. Estrutura e significado das colunas

In [6]:
# ------ Leitura do dicionário simplificado
opcoes_dicionario = [
    pasta_dados / 'DicionarioDados_GeracaoPorUsina.json',
]

caminho_dicionario = next(
    (arquivo for arquivo in opcoes_dicionario if arquivo.is_file()),
    None
)

if caminho_dicionario is None:
    raise FileNotFoundError(
        'Dicionário JSON não encontrado em data/raw/. '
        'Confira o nome do arquivo.'
    )

with caminho_dicionario.open(encoding='utf-8-sig') as arquivo:
    dicionario = json.load(arquivo)

tabela_dicionario = pd.DataFrame(dicionario['dicionario_simplificado'])
tabela_dicionario = tabela_dicionario[['codigo', 'descricao']]

display(tabela_dicionario)

,codigo,descricao
0,din_instante,Data de referência
1,id_subsistema,Subsistema da Usina
2,nom_subsistema,Nome do Subsistema
3,id_estado,Estado onde está localizado a Usina
4,nom_estado,Nome do Estado
5,cod_modalidadeoperacao,Modalidade de Operação da Usina
6,nom_tipousina,Tipo da Usina
7,nom_tipocombustivel,Tipo de Combustível
8,nom_usina,Nome da Usina
9,id_ons,Identificador ONS


Cada linha associa um conjunto a um instante. `id_ons` identifica o conjunto; `nom_usina` traz seu nome; `din_instante` contém o rótulo de data e hora; e `val_geracao` é a geração média informada em **MWmed**. No recorte, os registros seguem uma frequência horária.

## 3. Recorte inicial: Belmonte em janeiro de 2024

In [ ]:
dados.info()   

<class 'pandas.DataFrame'>
RangeIndex: 52632 entries, 0 to 52631
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   din_instante            52632 non-null  str    
 1   id_subsistema           52632 non-null  str    
 2   nom_subsistema          52632 non-null  str    
 3   id_estado               52632 non-null  str    
 4   nom_estado              52632 non-null  str    
 5   cod_modalidadeoperacao  52632 non-null  str    
 6   nom_tipousina           52632 non-null  str    
 7   nom_tipocombustivel     52632 non-null  str    
 8   nom_usina               52632 non-null  str    
 9   id_ons                  52632 non-null  str    
 10  ceg                     52632 non-null  str    
 11  val_geracao             52632 non-null  float64
dtypes: float64(1), str(11)
memory usage: 4.8 MB


In [7]:
# ------ Formato datetime
dados['din_instante'] = pd.to_datetime(dados['din_instante'])

# ------ Filtragem de Belmonte em janeiro de 2024
resultado = dados.loc[
    (dados['nom_usina'] == 'Conj. Belmonte') &
    (dados['din_instante'].dt.month == 1) &
    (dados['din_instante'].dt.year == 2024)
].copy()

# ------ Organização por data
resultado = resultado.sort_values('din_instante').reset_index(drop=True)

print('Identificação do recorte:')
display(resultado[['nom_usina', 'id_ons']].drop_duplicates())
display(resultado)

Identificação do recorte:


,nom_usina,id_ons
0,Conj. Belmonte,CJU_PEBELM


,din_instante,id_subsistema,nom_subsistema,id_estado,nom_estado,cod_modalidadeoperacao,nom_tipousina,nom_tipocombustivel,nom_usina,id_ons,ceg,val_geracao
0,2024-01-01 00:00:00,NE,NORDESTE,PE,PERNAMBUCO,Conjunto de Usinas,FOTOVOLTAICA,Fotovoltaica,Conj. Belmonte,CJU_PEBELM,-,0.0
1,2024-01-01 01:00:00,NE,NORDESTE,PE,PERNAMBUCO,Conjunto de Usinas,FOTOVOLTAICA,Fotovoltaica,Conj. Belmonte,CJU_PEBELM,-,0.0
2,2024-01-01 02:00:00,NE,NORDESTE,PE,PERNAMBUCO,Conjunto de Usinas,FOTOVOLTAICA,Fotovoltaica,Conj. Belmonte,CJU_PEBELM,-,0.0
3,2024-01-01 03:00:00,NE,NORDESTE,PE,PERNAMBUCO,Conjunto de Usinas,FOTOVOLTAICA,Fotovoltaica,Conj. Belmonte,CJU_PEBELM,-,0.0
4,2024-01-01 04:00:00,NE,NORDESTE,PE,PERNAMBUCO,Conjunto de Usinas,FOTOVOLTAICA,Fotovoltaica,Conj. Belmonte,CJU_PEBELM,-,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
739,2024-01-31 19:00:00,NE,NORDESTE,PE,PERNAMBUCO,Conjunto de Usinas,FOTOVOLTAICA,Fotovoltaica,Conj. Belmonte,CJU_PEBELM,-,0.0
740,2024-01-31 20:00:00,NE,NORDESTE,PE,PERNAMBUCO,Conjunto de Usinas,FOTOVOLTAICA,Fotovoltaica,Conj. Belmonte,CJU_PEBELM,-,0.0
741,2024-01-31 21:00:00,NE,NORDESTE,PE,PERNAMBUCO,Conjunto de Usinas,FOTOVOLTAICA,Fotovoltaica,Conj. Belmonte,CJU_PEBELM,-,0.0
742,2024-01-31 22:00:00,NE,NORDESTE,PE,PERNAMBUCO,Conjunto de Usinas,FOTOVOLTAICA,Fotovoltaica,Conj. Belmonte,CJU_PEBELM,-,0.0


## 4. Qualidade: datas, duplicatas, ausências e geração

In [ ]:
# ■■■■■■■■■■■■■■ QUALIDADE DO RECORTE ■■■■■■■■■■■■■■

# ------ Datas do recorte
datas = pd.to_datetime(resultado['din_instante'], errors='coerce')

print('Quantidade de registros:', len(resultado))
print('Primeira data:', datas.min())
print('Última data:', datas.max())
print('Datas inválidas ou ausentes no recorte:', datas.isna().sum())


# ------ Duplicatas pela chave: conjunto + instante
duplicatas = resultado[
    resultado.duplicated(
        subset=['id_ons', 'din_instante'],
        keep=False
    )
]

print('\nLinhas envolvidas em duplicatas:', len(duplicatas))

if not duplicatas.empty:
    display(duplicatas.sort_values(['id_ons', 'din_instante']))


# ------ Horas ausentes em janeiro de 2024
horas_esperadas = pd.date_range(
    start='2024-01-01 00:00:00',
    end='2024-02-01 00:00:00',
    freq='h',
    inclusive='left'
)

horas_observadas = pd.DatetimeIndex(datas.dropna().unique())
horas_ausentes = horas_esperadas.difference(horas_observadas)

print('\nHoras esperadas:', len(horas_esperadas))
print('Horas ausentes:', len(horas_ausentes))

if len(horas_ausentes) > 0:
    display(horas_ausentes)


# ------ Verificação dos valores de geração
geracao = pd.to_numeric(resultado['val_geracao'], errors='coerce')

print('\nGeração ausente ou não numérica:', geracao.isna().sum())
print('Valores de geração negativos:', (geracao < 0).sum())
print('Valores de geração iguais a zero:', (geracao == 0).sum())

Quantidade de registros: 744
Primeira data: 2024-01-01 00:00:00
Última data: 2024-01-31 23:00:00
Datas inválidas ou ausentes no recorte: 0

Linhas envolvidas em duplicatas: 0

Horas esperadas: 744
Horas ausentes: 0

Geração ausente ou não numérica: 0
Valores de geração negativos: 0
Valores de geração iguais a zero: 333


### Regra para duplicatas

Se for encontrado linhas exatamente iguais, registrar a ocorrência e manter apenas uma delas na base tratada. Se houver linhas com o mesmo id_ons e horário, mas com dados diferentes, mantê-las e investigar antes de decidir qual versão usar.

### Sequências de geração zero

Os zeros podem ocorrer naturalmente em períodos sem geração solar. A tabela abaixo identifica sua duração em registros horários e seus horários.

In [9]:
# ------ Identifica sequências consecutivas de geração zero
dados_zeros = resultado.copy()
dados_zeros['geracao_zero'] = dados_zeros['val_geracao'].eq(0)

nova_sequencia = (
    dados_zeros['geracao_zero'].ne(dados_zeros['geracao_zero'].shift())
    | dados_zeros['din_instante'].diff().ne(pd.Timedelta(hours=1))
)
dados_zeros['sequencia'] = nova_sequencia.cumsum()

sequencias_zero = (
    dados_zeros.loc[dados_zeros['geracao_zero']]
    .groupby('sequencia')
    .agg(
        inicio=('din_instante', 'min'),
        fim=('din_instante', 'max'),
        quantidade_registros=('din_instante', 'size')
    )
    .reset_index(drop=True)
)

print('Quantidade de sequências:', len(sequencias_zero))
display(sequencias_zero)

Quantidade de sequências: 32


,inicio,fim,quantidade_registros
0,2024-01-01 00:00:00,2024-01-01 06:00:00,7
1,2024-01-01 18:00:00,2024-01-02 04:00:00,11
2,2024-01-02 18:00:00,2024-01-03 05:00:00,12
3,2024-01-03 18:00:00,2024-01-04 04:00:00,11
4,2024-01-04 18:00:00,2024-01-05 04:00:00,11
5,2024-01-05 18:00:00,2024-01-06 04:00:00,11
6,2024-01-06 18:00:00,2024-01-07 04:00:00,11
7,2024-01-07 18:00:00,2024-01-08 04:00:00,11
8,2024-01-08 18:00:00,2024-01-09 06:00:00,13
9,2024-01-09 19:00:00,2024-01-10 04:00:00,10


Normal, dentro do esperado. 

## 5. Base completa

In [ ]:
# ■■■ QUALIDADE DA BASE COMPLETA ■■■

# ------ Horários esperados entre 2024 e 2025
horarios_esperados = pd.date_range(
    start='2024-01-01 00:00:00',
    end='2025-12-31 23:00:00',
    freq='h'
)

# ------ Verificação de cada conjunto
resumo = []

for id_conjunto, grupo in dados.groupby('id_ons'):
    horarios_observados = pd.DatetimeIndex(
        grupo['din_instante'].dropna().unique()
    )

    horarios_ausentes = horarios_esperados.difference(
        horarios_observados
    )

    resumo.append({
        'id_ons': id_conjunto,
        'nome': grupo['nom_usina'].iloc[0],
        'registros': len(grupo),
        'horas_esperadas': len(horarios_esperados),
        'horas_ausentes': len(horarios_ausentes),
        'linhas_duplicadas': grupo.duplicated(
            subset=['din_instante'],
            keep=False
        ).sum(),
        'geracao_ausente': grupo['val_geracao'].isna().sum(),
        'geracao_negativa': (grupo['val_geracao'] < 0).sum()
    })

resumo_qualidade = pd.DataFrame(resumo)

display(resumo_qualidade)

,id_ons,nome,registros,horas_esperadas,horas_ausentes,linhas_duplicadas,geracao_ausente,geracao_negativa
0,CJU_PEBELM,Conj. Belmonte,17544,17544,0,0,0,0
1,CJU_PEBRI,Conj. Brígida,17544,17544,0,0,0,0
2,CJU_PECIR,Conj. Ciranda,17544,17544,0,0,0,0


In [ ]:
# BANCO SQLITE

## 7. Conclusões e próximos passos

### Conclusões

Na versão do CSV analisada (SHA-256 929c2731942bfd4fd814044620d4d1283dd8dcb9cabb94eeeadc9d984d71d119), examinei os dados de Belmonte, Brígida e Ciranda entre 2024 e 2025.

Cada conjunto possui 17.544 registros, correspondentes às 17.544 horas esperadas no período. Não foram encontradas horas ausentes, linhas duplicadas por conjunto e instante, nem valores de geração ausentes ou negativos.